# Advanced Session: Check Generated Crystals with SevenNet

AI for Materials Science — Hands-on session 8 (optional, nothing to submit)

In class, the four checks told us whether a generated crystal is *plausible*. They did not tell us whether it is
*stable*. Here you answer that with SevenNet, the machine-learning interatomic potential from Hands-on session 7:
relax each generated structure, and compare its energy with the known Li–P–S phases relaxed with the same model.

A **T4 GPU** runtime is recommended. On a CPU the notebook relaxes only three structures and skips the MD part.

## Plan

- **0. Setup:** install SevenNet and download the data.
- **A. Structures:** one of the TA's MatterGen sets, or your own from class.
- **B. Relax with SevenNet:** positions and cell, until the forces are small.
- **C. Energy above the hull:** compare with the known phases on the same footing.
- **D. Optional:** a short molecular dynamics run to watch the Li ions move.

**On a shared computer:** use *File → Save a copy in Drive* to keep your work, and at the end use *Runtime → Disconnect and delete runtime*, then sign out of Google.

## 0. Setup

### 0-1. Install the libraries

Run this cell first. It installs SevenNet (with its dependencies), pymatgen and ASE. Colab already includes PyTorch.

In [ ]:
!pip install -q sevenn==0.13.0 pymatgen==2026.10.24 ase==3.29.0

### 0-2. Download the course material

This cell copies the course repository into a folder called `MS49900-AI4M`. The notebook reads these files from its `Data/` folder:

- `mattergen_pregenerated/`: the TA's MatterGen structures (A-1 lists the sets).
- `mp_LiPS_references.json.gz`: known Li–P–S(–Cl, –Br) phases from the Materials Project.
- `mp_LiPS_sevennet_energies.csv`: those phases relaxed by the TA with SevenNet, with the same settings as below.

If git says the destination already exists, it leaves that folder unchanged and does not update the files. If a later cell reports a missing file, check the contents of `MS49900-AI4M/Data/`.

In [ ]:
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. Import the libraries

The next cell imports the tools used throughout the notebook.

In [ ]:
import gzip, json, math, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ase.filters import FrechetCellFilter
from ase.optimize import FIRE
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase import units
from pymatgen.core import Composition, Structure
from pymatgen.analysis.phase_diagram import PDEntry, PhaseDiagram
from pymatgen.io.ase import AseAtomsAdaptor
import sevenn
from sevenn.calculator import SevenNetCalculator

### 0-4. Set the file paths

The settings must match those used for the reference energies; the cell stops if they do not.

In [ ]:
DATA_DIR = Path("MS49900-AI4M/Data")
PREGEN = DATA_DIR / "mattergen_pregenerated"
MODEL, FMAX, STEPS = "7net-0", 0.05, 300   ## the same settings as the reference energies
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ref_prov = json.loads((DATA_DIR / "mp_LiPS_sevennet_energies_provenance.json").read_text())
assert (ref_prov["model"], ref_prov["fmax"], ref_prov["steps"]) == (MODEL, FMAX, STEPS), "settings differ from the reference energies"
print("device:", DEVICE, "| sevenn", sevenn.__version__, "| reference energies made with sevenn", ref_prov["sevenn_version"])

## A. Structures

### A-1. Choose a set

`SET` picks one of the TA's sets; the cell prints which sets are in `Data/`. To use your own structures from class,
download `mg_live/<condition>/generated_crystals_cif.zip` from that runtime, upload it here with the file panel, and
set `MY_ZIP` to its name.

In [ ]:
def load_cifs(zip_path):
    """Read every CIF in a MatterGen output zip as a pymatgen Structure."""
    with zipfile.ZipFile(zip_path) as zf:
        names = sorted(n for n in zf.namelist() if n.endswith(".cif"))
        return [Structure.from_str(zf.read(n).decode(), fmt="cif") for n in names]


TA_SETS = json.loads((DATA_DIR / "mattergen_pregenerated_provenance.json").read_text())["sets"]
print("TA sets:", {k: f'{v["n_structures"]} structures ({v["device"]})' for k, v in TA_SETS.items()})
SET = "C3"
MY_ZIP = None                       ## for example Path("generated_crystals_cif.zip")
zip_path = Path(MY_ZIP) if MY_ZIP else PREGEN / SET / "generated_crystals_cif.zip"
if not zip_path.exists():
    raise FileNotFoundError(f"{zip_path} not found: choose SET from {sorted(TA_SETS)}, or upload your zip and set MY_ZIP.")
structures = load_cifs(zip_path)
N_MAX = len(structures) if DEVICE == "cuda" else 3
structures = structures[:N_MAX]
print(len(structures), "structures:", [s.composition.reduced_formula for s in structures])

## B. Relax with SevenNet

### B-1. The calculator

`SevenNetCalculator` predicts energies and forces with the pretrained `7net-0` model, which was trained on the
Materials Project trajectory data (MPtrj), so its energies are comparable with those of Materials Project structures
relaxed with the same model.

In [ ]:
calc = SevenNetCalculator(MODEL, device=DEVICE)
print(calc)

### B-2. Relax positions and cell

FIRE moves the atoms, and `FrechetCellFilter` also lets the cell shape and volume change, until every force is below
`FMAX` eV/Å or `STEPS` steps have passed. A generated structure that is far from any minimum moves a lot here; that
alone says something about the generator.

In [ ]:
def relax(structure):
    atoms = AseAtomsAdaptor.get_atoms(structure)
    atoms.calc = calc
    opt = FIRE(FrechetCellFilter(atoms), logfile=None)
    converged = bool(opt.run(fmax=FMAX, steps=STEPS))
    return AseAtomsAdaptor.get_structure(atoms), atoms.get_potential_energy() / len(atoms), converged, opt.nsteps

rows, relaxed = [], []
t0 = time.time()
for s in structures:
    r, e, ok, n = relax(s)
    relaxed.append(r)
    rows.append({"formula": s.composition.reduced_formula, "atoms": len(s), "energy_eV_per_atom": round(e, 4),
                 "converged": ok, "steps": n, "volume_change_%": round(100 * (r.volume / s.volume - 1), 1)})
gen_table = pd.DataFrame(rows)
print(f"relaxed {len(rows)} structures in {time.time() - t0:.0f} s")
gen_table

## C. Energy above the hull, with the same model

### C-1. The known phases

The TA relaxed every known phase in `mp_LiPS_references.json.gz` with the same model and settings. From these
energies pymatgen builds the **convex hull**: the lowest energy any mix of known phases can reach at each
composition. A new structure is promising if its energy lies on or close to the hull.

In [ ]:
refs = pd.read_csv(DATA_DIR / "mp_LiPS_sevennet_energies.csv")
ref_entries = [PDEntry(Composition(r.composition), r.energy_eV_per_atom * r.nsites, name=r.material_id)
               for r in refs.itertuples()]
pd_hull = PhaseDiagram(ref_entries)
print(len(ref_entries), "reference entries | elements:", [str(e) for e in pd_hull.elements])

### C-2. Where do the generated structures sit?

`e_hull_7net` is the energy above the hull in eV/atom. A negative value would mean the structure is more stable than
every known phase at that composition, which is rare and worth a closer look (or a sign of a model error).
`e_vs_known` compares with the lowest known phase of the same formula, when there is one.

In [ ]:
hull_elements = set(pd_hull.elements)
lowest_known = refs.assign(f=refs.composition.map(lambda c: Composition(c).reduced_formula)) \
                   .groupby("f").energy_eV_per_atom.min()
e_hull, e_known = [], []
for r, row in zip(relaxed, rows):
    comp = r.composition
    if set(comp.elements) <= hull_elements:
        entry = PDEntry(comp, row["energy_eV_per_atom"] * len(r))
        e_hull.append(round(pd_hull.get_e_above_hull(entry, allow_negative=True), 4))
    else:
        e_hull.append(np.nan)
    f = comp.reduced_formula
    e_known.append(round(row["energy_eV_per_atom"] - lowest_known[f], 4) if f in lowest_known else np.nan)
gen_table["e_hull_7net"], gen_table["e_vs_known"] = e_hull, e_known
if not gen_table.converged.all():   ## kept for inspection, but their energies are provisional
    print(f"Not converged within {STEPS} steps (provisional, left out of the ranking and of part D):",
          list(gen_table.formula[~gen_table.converged]))
gen_table.sort_values(["converged", "e_hull_7net"], ascending=[False, True])

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.5))
t = gen_table.sort_values(["converged", "e_hull_7net"], ascending=[False, True]).reset_index(drop=True)
ok = t.converged
ax.bar(t.index[ok], t.e_hull_7net[ok], color=["tab:green" if v <= 0.05 else "tab:gray" for v in t.e_hull_7net[ok]])
if (~ok).any():
    ax.bar(t.index[~ok], t.e_hull_7net[~ok], color="none", edgecolor="tab:gray", hatch="//",
           label="not converged (provisional)")
ax.axhline(0.05, ls="--", c="k", lw=1, label="0.05 eV/atom (the condition)")
ax.set_xticks(t.index, t.formula, rotation=60, ha="right"); ax.set_ylabel("energy above hull (eV/atom)")
ax.set_title(f"{SET if MY_ZIP is None else MY_ZIP}: stability after relaxation with {MODEL}"); ax.legend()
plt.show()

### C-3. How far can we trust this?

An MLIP energy has errors of a few tens of meV/atom, so values within about 0.05 eV/atom of the hull are all
candidates, not proofs. The hull is also only as complete as its reference pool: it is built from the 88 phases in our
file (Li–P–S(–Cl, –Br) phases within 0.1 eV/atom of the Materials Project hull), not from every known compound, so a
low value is not a proof of stability. The usual next step is a DFT relaxation of the best few structures, and then the question
that matters for an electrolyte: do the Li ions move? Part D gives a first, rough look at that.

## D. Optional: a short molecular dynamics run (GPU)

This runs 2 ps of Langevin dynamics at 600 K for the most stable converged Li-containing structure from C-2 (in a supercell
with at least 60 atoms) and plots the mean squared displacement (MSD) of the Li ions. A steadily rising MSD means
the Li ions hop between sites; a flat one means they only vibrate. 2 ps is far too short for a diffusion
coefficient; treat this as a demonstration.

In [ ]:
t = gen_table.assign(i=range(len(gen_table)))
md_candidates = t[t.converged & t.e_hull_7net.notna() & t.formula.str.contains("Li")].sort_values("e_hull_7net")
if DEVICE != "cuda":
    print("Skipped on a CPU runtime: switch to a T4 GPU to run the MD part.")
elif md_candidates.empty:
    print("Skipped: this set has no converged Li-containing structure with a hull energy. Choose another SET.")
else:
    s = relaxed[int(md_candidates.i.iloc[0])].copy()
    s.make_supercell([max(1, math.ceil((60 / len(s)) ** (1 / 3)))] * 3)
    atoms = AseAtomsAdaptor.get_atoms(s)
    atoms.calc = calc
    MaxwellBoltzmannDistribution(atoms, temperature_K=600)
    dyn = Langevin(atoms, 2 * units.fs, temperature_K=600, friction=0.02)
    li = [a.index for a in atoms if a.symbol == "Li"]
    start = atoms.get_positions()[li].copy()
    msd = []
    def record():
        msd.append(((atoms.get_positions()[li] - start) ** 2).sum(axis=1).mean())
    dyn.attach(record, interval=10)
    t0 = time.time()
    dyn.run(1000)
    print(f"{len(atoms)} atoms, 2 ps in {time.time() - t0:.0f} s")
    plt.figure(figsize=(5, 3.2)); plt.plot(np.arange(len(msd)) * 0.02, msd)
    plt.xlabel("time (ps)"); plt.ylabel("Li MSD (Å²)"); plt.title(f"{s.composition.reduced_formula} at 600 K")
    plt.show()

## Data sources

- **MatterGen structures:** `Data/mattergen_pregenerated/`, generated by the TA with MatterGen: C. Zeni et al.,
  *Nature* 639, 624–632 (2025), https://doi.org/10.1038/s41586-025-08628-5; code and checkpoints
  https://github.com/microsoft/mattergen (MIT License).
- **Materials Project:** `Data/mp_LiPS_references.json.gz`, accessed 2026-11-04, licensed under CC BY 4.0
  (https://creativecommons.org/licenses/by/4.0/). Cite: A. Jain et al., *APL Materials* 1, 011002 (2013),
  https://doi.org/10.1063/1.4812323. `Data/mp_LiPS_sevennet_energies.csv` holds these phases relaxed by the TA with
  SevenNet; see its provenance file.
- **SevenNet:** Y. Park, J. Kim, S. Hwang, S. Han, *Journal of Chemical Theory and Computation* 20, 4857–4868 (2024),
  https://doi.org/10.1021/acs.jctc.4c00190.
- **pymatgen:** S. P. Ong et al., *Computational Materials Science* 68, 314 (2013). **ASE:** A. H. Larsen et al.,
  *Journal of Physics: Condensed Matter* 29, 273002 (2017).